---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — include notebook name, cell number, and what went wrong.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first.*

### 💾 Save your own copy
> **File → Save a copy in Drive** — saves a personal editable copy to your Google Drive.
> Do this before making edits, otherwise changes are lost when the session ends.

# 🔍 Subset Selection
**ISLP Chapter 6 · Pattern Recognition for the Rest of Us**

> With p predictors there are 2ᵖ possible models. Which features actually matter? Subset selection answers that question systematically.

### What you'll learn
- Why variable selection matters (overfitting, interpretability, speed)
- Forward stepwise selection — the greedy approach
- Backward stepwise selection — prune from the full model
- Model selection criteria: CV MSE, AIC, BIC, Adjusted R²

### Time: ~45 minutes

In [ ]:
# ⚠️  Run this cell first — sets up data and imports
# Tip: Runtime → Run all  (runs everything top to bottom)

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from itertools import combinations
import statsmodels.api as sm

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

# ── Load Hitters dataset (ISLP) ───────────────────────────────────────────────
try:
    hitters = pd.read_csv('https://www.statlearning.com/s/Hitters.csv')
    print(f'✓ Hitters.csv (ISLP): {hitters.shape}')
except Exception:
    hitters = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/Hitters.csv')
    print(f'✓ Hitters.csv (fallback): {hitters.shape}')

hitters = hitters.dropna()
print(f'  After dropna: {hitters.shape}')

# ── Prepare feature matrix ────────────────────────────────────────────────────
# Encode categorical columns; log-transform Salary (right-skewed)
hitters_enc = pd.get_dummies(hitters, columns=['League','Division','NewLeague'],
                              drop_first=True, dtype=float)
hitters_enc['log_salary'] = np.log(hitters_enc['Salary'])
hitters_enc = hitters_enc.drop(columns=['Salary'])

target   = 'log_salary'
features = [c for c in hitters_enc.columns if c != target]
X = hitters_enc[features].values.astype(float)
y = hitters_enc[target].values

# Train / test split — hold out 20% for final evaluation
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'  Features ({len(features)}): {features}')
print(f'  X shape: {X.shape}  |  train: {X_tr.shape[0]}  test: {X_te.shape[0]}')
print(f'  Target: log(Salary), mean={y.mean():.3f}, std={y.std():.3f}')
print(f'  Python {sys.version.split()[0]} | pandas {pd.__version__}')
print('✓ Setup complete')


## 📐 Part 1 — Why Variable Selection?

With **p ≈ 19** predictors in Hitters, there are 2¹⁹ = 524,288 possible models. We can't fit them all.

**Three main approaches:**
1. **Best subset** — fit all 2ᵖ models, pick best by CV/AIC/BIC (only feasible for small p)
2. **Forward stepwise** — start empty, greedily add the best feature at each step
3. **Backward stepwise** — start full, greedily drop the least useful feature at each step

**Why not just use all features?**
- Overfitting — more predictors → lower training MSE, but higher test MSE
- Interpretability — a 5-variable model is easier to explain than 19
- Multicollinearity — correlated predictors inflate coefficient variance

In [ ]:
# ── Motivating example: overfitting with too many features ───────────────────
lr = LinearRegression()

# All features vs 3 features
cv_all  = -cross_val_score(lr, X_tr, y_tr, cv=10,
                            scoring='neg_mean_squared_error').mean()
top3_names = ['AtBat','Hits','Walks']
top3_idx   = [features.index(f) for f in top3_names if f in features]
cv_top3 = -cross_val_score(lr, X_tr[:, top3_idx], y_tr, cv=10,
                             scoring='neg_mean_squared_error').mean()

print(f"All {len(features)} features — 10-fold CV MSE: {cv_all:.4f}")
print(f"3 features         — 10-fold CV MSE: {cv_top3:.4f}")
print()

# Show train vs test MSE as we add noise features
np.random.seed(42)
sizes, train_mses, test_mses = [], [], []
for k in range(1, 26):
    if k <= X_tr.shape[1]:
        Xk_tr = X_tr[:, :k]
        Xk_te = X_te[:, :k]
    else:
        n_noise = k - X_tr.shape[1]
        Xk_tr = np.hstack([X_tr, np.random.randn(X_tr.shape[0], n_noise)])
        Xk_te = np.hstack([X_te, np.random.randn(X_te.shape[0], n_noise)])
    lr.fit(Xk_tr, y_tr)
    sizes.append(k)
    train_mses.append(mean_squared_error(y_tr, lr.predict(Xk_tr)))
    test_mses.append(mean_squared_error(y_te, lr.predict(Xk_te)))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sizes, train_mses, 'o-', color='#1e5fa8', lw=2, label='Train MSE')
ax.plot(sizes, test_mses,  'o-', color='#e85d20', lw=2, label='Test MSE')
ax.axvline(X_tr.shape[1], color='grey', lw=1, ls='--', alpha=0.7,
           label='Real features end')
ax.set_xlabel('Number of features (real + noise)')
ax.set_ylabel('MSE (log salary)')
ax.set_title('Train MSE always falls — Test MSE reveals overfitting')
ax.legend()
plt.tight_layout()
plt.show()

print("📌 Train MSE falls monotonically — even noise features appear helpful.")
print("   Test MSE rises once noise dominates. Variable selection finds the sweet spot.")


## 🔬 Part 2 — Forward Stepwise Selection

Start with no features. At each step, add the single feature that most reduces CV MSE.

**Time complexity:** O(p²) fits — much faster than 2ᵖ for best subset.

In [ ]:
# ── Forward stepwise selection ────────────────────────────────────────────────
def forward_stepwise(X, y, feature_names, cv=10):
    n_feat    = X.shape[1]
    selected  = []
    remaining = list(range(n_feat))
    results   = []
    lr = LinearRegression()

    for step in range(n_feat):
        best_mse, best_feat = np.inf, None
        for feat in remaining:
            candidate = selected + [feat]
            mse = -cross_val_score(lr, X[:, candidate], y, cv=cv,
                                   scoring='neg_mean_squared_error').mean()
            if mse < best_mse:
                best_mse, best_feat = mse, feat
        selected.append(best_feat)
        remaining.remove(best_feat)
        results.append({
            'n_features':  step + 1,
            'last_added':  feature_names[best_feat],
            'cv_mse':      best_mse,
            'feature_idx': selected.copy()
        })

    return pd.DataFrame(results)

print('Running forward stepwise (this takes ~20 seconds)...')
fwd = forward_stepwise(X_tr, y_tr, features)

print()
print(fwd[['n_features','last_added','cv_mse']].to_string(index=False))
best_row_fwd  = fwd.loc[fwd['cv_mse'].idxmin()]
best_n_fwd    = int(best_row_fwd['n_features'])
best_idx_fwd  = best_row_fwd['feature_idx']
best_feat_fwd = [features[i] for i in best_idx_fwd]
print(f"\n📌 Best model: {best_n_fwd} features (CV MSE = {best_row_fwd['cv_mse']:.4f})")
print(f"   {best_feat_fwd}")


## 📊 Part 3 — Backward Stepwise Selection

Start with ALL features. At each step, drop the feature whose removal least increases CV MSE.

Use **forward** when p is large (p > n makes the full model undefined).
Use **backward** when you want to start from the full model and prune.

In [ ]:
# ── Backward stepwise selection ───────────────────────────────────────────────
def backward_stepwise(X, y, feature_names, cv=10):
    n_feat   = X.shape[1]
    selected = list(range(n_feat))
    results  = []
    lr = LinearRegression()

    # Full model
    full_mse = -cross_val_score(lr, X[:, selected], y, cv=cv,
                                 scoring='neg_mean_squared_error').mean()
    results.append({'n_features': n_feat, 'last_removed': 'None (full model)',
                    'cv_mse': full_mse, 'feature_idx': selected.copy()})

    for _ in range(n_feat - 1):
        best_mse, worst_feat = np.inf, None
        for feat in selected:
            candidate = [f for f in selected if f != feat]
            if not candidate:
                continue
            mse = -cross_val_score(lr, X[:, candidate], y, cv=cv,
                                   scoring='neg_mean_squared_error').mean()
            if mse < best_mse:
                best_mse, worst_feat = mse, feat
        selected.remove(worst_feat)
        results.append({'n_features': len(selected),
                        'last_removed': feature_names[worst_feat],
                        'cv_mse': best_mse,
                        'feature_idx': selected.copy()})

    return pd.DataFrame(results).sort_values('n_features').reset_index(drop=True)

print('Running backward stepwise (this takes ~20 seconds)...')
bwd = backward_stepwise(X_tr, y_tr, features)

print()
print(bwd[['n_features','last_removed','cv_mse']].to_string(index=False))
best_row_bwd  = bwd.loc[bwd['cv_mse'].idxmin()]
best_n_bwd    = int(best_row_bwd['n_features'])
best_feat_bwd = [features[i] for i in best_row_bwd['feature_idx']]
print(f"\n📌 Best model: {best_n_bwd} features (CV MSE = {best_row_bwd['cv_mse']:.4f})")
print(f"   {best_feat_bwd}")

# ── Compare forward vs backward ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(fwd['n_features'], fwd['cv_mse'], 'o-', color='#1e5fa8', lw=2,
        label='Forward stepwise')
ax.plot(bwd['n_features'], bwd['cv_mse'], 's--', color='#e85d20', lw=2,
        label='Backward stepwise')
ax.axvline(best_n_fwd, color='#1e5fa8', lw=1.2, ls=':',
           label=f'Fwd best: {best_n_fwd} features')
ax.axvline(best_n_bwd, color='#e85d20', lw=1.2, ls=':',
           label=f'Bwd best: {best_n_bwd} features')
ax.set_xlabel('Number of features in model')
ax.set_ylabel('10-fold CV MSE (log salary)')
ax.set_title('Forward vs Backward Stepwise — Hitters Dataset')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("\n📌 Forward and backward often agree but can select different subsets.")
print("   Neither guarantees the globally optimal subset — use as starting points.")


## ✅ Part 4 — Model Selection Criteria: AIC, BIC, Adjusted R²

Instead of rerunning CV for every model size, information criteria penalise complexity analytically:

| Criterion | Penalty | Prefers |
|-----------|---------|---------|
| **AIC** | 2p | Prediction accuracy |
| **BIC** | p · ln(n) | Parsimony (heavier penalty) |
| **Adj R²** | Penalises p/(n−p−1) | Avoids spurious gains |

**BIC** selects smaller models than AIC when n > 7 (since ln(n) > 2).
**Adj R²** increases only when a new predictor reduces error more than expected by chance.

In [ ]:
# ── AIC / BIC / Adj R² along the forward stepwise path ──────────────────────
criteria = []
for k in range(1, len(features) + 1):
    idx_k = fwd.loc[fwd['n_features'] == k, 'feature_idx'].values[0]
    Xk    = sm.add_constant(X_tr[:, idx_k])
    mk    = sm.OLS(y_tr, Xk).fit()
    criteria.append({
        'n_features': k,
        'AIC':    mk.aic,
        'BIC':    mk.bic,
        'Adj_R2': mk.rsquared_adj,
        'CV_MSE': fwd.loc[fwd['n_features'] == k, 'cv_mse'].values[0]
    })

crit_df = pd.DataFrame(criteria)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
ax_flat = axes.ravel()

for ax, col, color in zip(ax_flat[:3], ['AIC','BIC','Adj_R2'],
                           ['#1e5fa8','#e85d20','#1a7a45']):
    ax.plot(crit_df['n_features'], crit_df[col], 'o-', color=color, lw=2)
    best = (crit_df[col].idxmin() if col != 'Adj_R2'
            else crit_df[col].idxmax())
    best_n = int(crit_df.loc[best, 'n_features'])
    ax.axvline(best_n, color='grey', lw=1.2, ls='--',
               label=f'Best: {best_n} features')
    ax.set_xlabel('Number of features'); ax.set_ylabel(col)
    ax.set_title(col); ax.legend(fontsize=9)

ax_flat[3].plot(crit_df['n_features'], crit_df['CV_MSE'],
                'o-', color='#c0392b', lw=2)
best_cv_n = int(crit_df.loc[crit_df['CV_MSE'].idxmin(), 'n_features'])
ax_flat[3].axvline(best_cv_n, color='grey', lw=1.2, ls='--',
                   label=f'Best: {best_cv_n} features')
ax_flat[3].set_xlabel('Number of features')
ax_flat[3].set_ylabel('10-fold CV MSE')
ax_flat[3].set_title('CV MSE (reference)')
ax_flat[3].legend(fontsize=9)

plt.suptitle('Model Selection Criteria — Forward Stepwise Path', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Optimal model sizes by criterion:")
print(f"  AIC:    {int(crit_df.loc[crit_df['AIC'].idxmin(),'n_features'])} features")
print(f"  BIC:    {int(crit_df.loc[crit_df['BIC'].idxmin(),'n_features'])} features")
print(f"  Adj R²: {int(crit_df.loc[crit_df['Adj_R2'].idxmax(),'n_features'])} features")
print(f"  CV MSE: {best_cv_n} features")
print("\n📌 BIC selects fewer features than AIC — its penalty grows with n.")
print("   When criteria disagree: prefer BIC for interpretation, CV for prediction.")


## 💼 Exercise

**Task 1 — Final model evaluation:** Fit the forward stepwise model (`best_idx_fwd` features) on the full training set and evaluate on the test set (`X_te`, `y_te`). Report RMSE in both log-salary and dollar terms.

**Task 2 — Compare to Lasso:** Import `LassoCV` and fit on `X_tr`, `y_tr`. How many features does it select (non-zero coefficients)? Compare test RMSE to the stepwise model.

**Task 3 — Do forward and backward agree?** Compare `best_feat_fwd` and `best_feat_bwd`. Which features appear in one but not the other? What does disagreement imply?

In [ ]:
# ── Exercise workspace ───────────────────────────────────────────────────────
# Variables available:
#   X_tr, X_te, y_tr, y_te, features
#   fwd       — forward stepwise results DataFrame
#   best_idx_fwd, best_feat_fwd, best_n_fwd
#   bwd       — backward stepwise results DataFrame
#   best_idx_bwd, best_feat_bwd, best_n_bwd

# Task 1 — Test evaluation of forward stepwise model
best_model  = LinearRegression().fit(X_tr[:, best_idx_fwd], y_tr)
y_pred_log  = best_model.predict(X_te[:, best_idx_fwd])
rmse_log    = np.sqrt(mean_squared_error(y_te, y_pred_log))
rmse_dollar = np.sqrt(mean_squared_error(np.exp(y_te), np.exp(y_pred_log)))
print(f"Forward stepwise ({best_n_fwd} features):")
print(f"  Test RMSE (log scale): {rmse_log:.4f}")
print(f"  Test RMSE ($):         ${rmse_dollar:,.0f}")
print()

# Task 2 — Lasso comparison
from sklearn.linear_model import LassoCV
lasso = LassoCV(cv=10, random_state=42).fit(X_tr, y_tr)
n_nonzero = np.sum(lasso.coef_ != 0)
lasso_rmse = np.sqrt(mean_squared_error(y_te, lasso.predict(X_te)))
print(f"LassoCV: {n_nonzero} non-zero features, test RMSE (log) = {lasso_rmse:.4f}")

# Task 3 — Forward vs Backward agreement
fwd_set = set(best_feat_fwd)
bwd_set = set(best_feat_bwd)
print(f"\nForward only:  {sorted(fwd_set - bwd_set)}")
print(f"Backward only: {sorted(bwd_set - fwd_set)}")
print(f"Both agree:    {sorted(fwd_set & bwd_set)}")


In [ ]:
# @title 📝 Quiz — Subset Selection
# @markdown Answer each question, then run this cell.

# @markdown **Q1:** Why does train MSE always decrease as we add more features?
# @markdown **Q2:** What is the key difference between forward and backward stepwise selection?
# @markdown **Q3:** Why does BIC tend to select smaller models than AIC?
# @markdown **Q4:** When is best-subset selection infeasible, and what is the alternative?
# @markdown **Q5:** What is the advantage of cross-validation over AIC/BIC for model selection?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the submission cell for AI feedback")


In [ ]:
_NB_NAME_ = "Subset Selection"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f'Please grade my quiz answers for the "{_NB_TITLE}" notebook')
    print(f'from "Data Pattern Recognition for the Rest of Us" (based on ISLP).')
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why")
    print("  3. Give the ideal complete answer")
    print("  4. If wrong/partial, say which Part to revisit")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan")


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md)

---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*